# Task 3.3

This notebook answers all parts of Task 3.3 for the production of 5000 litres of amber beer.

In [19]:
from brew_utils import *
from scipy.optimize import linprog
import numpy as np
import pandas as pd

## 3.3.1 Decision Variables and Data

This section uses the same MILP structure as in Task 3.2, but now the concrete instance is fixed: 5000 litres of amber beer. There are no minimum quality constraints in Task 3.3, so only the volume, color, process, and ingredient constraints matter.

In [20]:
barley_cost = barley["cost"].to_numpy(dtype=float)
barley_ebc = barley["EBC"].to_numpy(dtype=float)
barley_q = barley["quality"].to_numpy(dtype=float)

malt_cost = malts["cost"].to_numpy(dtype=float)
malt_ebc = malts["EBC"].to_numpy(dtype=float)
malt_q = malts["quality"].to_numpy(dtype=float)

extract_cost = maltextracts["cost"].to_numpy(dtype=float)
extract_ebc = maltextracts["EBC"].to_numpy(dtype=float)
extract_q = maltextracts["quality"].to_numpy(dtype=float)

hop_cost = hops["cost"].to_numpy(dtype=float)
hop_q = hops["quality"].to_numpy(dtype=float)

yeast_cost = yeasts["cost"].to_numpy(dtype=float)
yeast_q = yeasts["quality"].to_numpy(dtype=float)

F_malt = float(maltprocess.loc[0, "fixedcost"])
v_malt = float(maltprocess.loc[0, "variablecost"])
F_mash = float(mashingprocess.loc[0, "fixedcost"])
v_mash = float(mashingprocess.loc[0, "variablecost"])

nB = len(barley)
nM = len(malts)
nE = len(maltextracts)
nH = len(hops)
nY = len(yeasts)

iB0 = 0
iM0 = iB0 + nB
iE0 = iM0 + nM
iH0 = iE0 + nE
iY0 = iH0 + nH
iZm = iY0 + nY
iZs = iZm + 1
nvars = iZs + 1

def slice_B():
    return slice(iB0, iB0 + nB)

def slice_M():
    return slice(iM0, iM0 + nM)

def slice_E():
    return slice(iE0, iE0 + nE)

def slice_H():
    return slice(iH0, iH0 + nH)

def slice_Y():
    return slice(iY0, iY0 + nY)

## 3.3.1 MILP Formulation

This block builds the MILP for 5000 litres of amber beer. The objective is to minimize total cost. The model includes ingredient balances, exact hop and yeast requirements, amber EBC bounds, and binary process variables for malting and mashing.

In [21]:
def build_lp_model(V_beer, EBC_min, EBC_max, z_fix=None):
    m_tot = malt_extract_eq_needed_for_beer(V_beer)
    H_tot = hops_needed_for_beer(V_beer)
    Y_tot = yeast_needed_for_beer(V_beer)
    gamma = (SG - 1.0) / 0.0344

    M_B = m_tot / BARLEY_TO_MALTEXTRACT_EQ
    M_M = m_tot / MALT_TO_MALTEXTRACT_EQ

    c = np.zeros(nvars)
    c[slice_B()] = barley_cost + v_malt + v_mash * BARLEY_TO_MALT
    c[slice_M()] = malt_cost + v_mash
    c[slice_E()] = extract_cost
    c[slice_H()] = hop_cost
    c[slice_Y()] = yeast_cost
    c[iZm] = F_malt
    c[iZs] = F_mash

    A_eq = []
    b_eq = []

    row = np.zeros(nvars)
    row[slice_B()] = BARLEY_TO_MALTEXTRACT_EQ
    row[slice_M()] = MALT_TO_MALTEXTRACT_EQ
    row[slice_E()] = 1.0
    A_eq.append(row)
    b_eq.append(m_tot)

    row = np.zeros(nvars)
    row[slice_H()] = 1.0
    A_eq.append(row)
    b_eq.append(H_tot)

    row = np.zeros(nvars)
    row[slice_Y()] = 1.0
    A_eq.append(row)
    b_eq.append(Y_tot)

    A_ub = []
    b_ub = []

    if np.isfinite(EBC_max):
        row = np.zeros(nvars)
        row[slice_B()] = gamma * BARLEY_TO_MALTEXTRACT_EQ * barley_ebc
        row[slice_M()] = gamma * MALT_TO_MALTEXTRACT_EQ * malt_ebc
        row[slice_E()] = gamma * extract_ebc
        A_ub.append(row)
        b_ub.append(EBC_max * m_tot)

    row = np.zeros(nvars)
    row[slice_B()] = -gamma * BARLEY_TO_MALTEXTRACT_EQ * barley_ebc
    row[slice_M()] = -gamma * MALT_TO_MALTEXTRACT_EQ * malt_ebc
    row[slice_E()] = -gamma * extract_ebc
    A_ub.append(row)
    b_ub.append(-EBC_min * m_tot)

    row = np.zeros(nvars)
    row[slice_B()] = 1.0
    row[iZm] = -M_B
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[slice_B()] = BARLEY_TO_MALT
    row[slice_M()] = 1.0
    row[iZs] = -M_M
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[iZm] = 1.0
    row[iZs] = -1.0
    A_ub.append(row)
    b_ub.append(0.0)

    bounds = [(0, None)] * nvars
    if z_fix is None:
        bounds[iZm] = (0, 1)
        bounds[iZs] = (0, 1)
    else:
        bounds[iZm] = (z_fix[0], z_fix[0])
        bounds[iZs] = (z_fix[1], z_fix[1])

    return {
        "c": c,
        "A_eq": np.array(A_eq, dtype=float),
        "b_eq": np.array(b_eq, dtype=float),
        "A_ub": np.array(A_ub, dtype=float),
        "b_ub": np.array(b_ub, dtype=float),
        "bounds": bounds,
        "m_tot": m_tot,
        "H_tot": H_tot,
        "Y_tot": Y_tot,
    }

## 3.3.2 Relaxation

The relaxation is obtained by allowing the binary process variables `z_malt` and `z_mash` to take any value in the interval `[0, 1]`. The code below solves that relaxed problem.

In [22]:
def solve_relaxation(V_beer, EBC_min, EBC_max):
    model = build_lp_model(V_beer, EBC_min, EBC_max, z_fix=None)
    res = linprog(
        c=model["c"],
        A_ub=model["A_ub"],
        b_ub=model["b_ub"],
        A_eq=model["A_eq"],
        b_eq=model["b_eq"],
        bounds=model["bounds"],
        method="highs",
    )
    return model, res

## 3.3.3 Why Most Ingredients Are Not Used

In a cost minimization LP, most variables are zero because the solver only keeps ingredients that help achieve the target at minimum cost. Without quality constraints, the model tends to use only the cheapest ingredients that satisfy the amber EBC interval and the process logic.

## 3.3.4 Rounding the Relaxation

To obtain a feasible MILP solution from the relaxation, we inspect the fractional process variables. Since the original problem requires binary process choices, we round them to a valid process configuration and then resolve the LP with those process choices fixed.

In [23]:
def solve_with_fixed_processes(V_beer, EBC_min, EBC_max, z_malt, z_mash):
    model = build_lp_model(V_beer, EBC_min, EBC_max, z_fix=(z_malt, z_mash))
    res = linprog(
        c=model["c"],
        A_ub=model["A_ub"],
        b_ub=model["b_ub"],
        A_eq=model["A_eq"],
        b_eq=model["b_eq"],
        bounds=model["bounds"],
        method="highs",
    )
    return model, res

def rounded_process_choice(relax_res):
    z_malt = int(round(relax_res.x[iZm]))
    z_mash = int(round(relax_res.x[iZs]))

    if z_malt > z_mash:
        z_mash = z_malt

    if (z_malt, z_mash) not in [(0, 0), (0, 1), (1, 1)]:
        z_malt, z_mash = 0, 1

    return z_malt, z_mash

## 3.3.5 Original MILP via Brute Force

The original MILP is solved by brute force over the valid process choices:
- `(0, 0)` only malt extract
- `(0, 1)` mashing but no malting
- `(1, 1)` both malting and mashing

For each case, the binary variables are fixed and the remaining problem is solved as a linear program. The cheapest feasible case is the MILP solution.

In [24]:
def solve_bruteforce(V_beer, EBC_min, EBC_max):
    candidates = [(0, 0), (0, 1), (1, 1)]
    best = None
    all_results = []

    for z_fix in candidates:
        model, res = solve_with_fixed_processes(V_beer, EBC_min, EBC_max, *z_fix)
        info = {
            "z_fix": z_fix,
            "success": res.success,
            "cost": res.fun if res.success else np.nan,
            "result": res,
            "model": model,
        }
        all_results.append(info)

        if res.success and (best is None or res.fun < best["cost"]):
            best = info

    return best, all_results

## Helpers for Reporting

These functions summarize the solution in a report-friendly way by showing the non-zero variables, the cost, the EBC value, and the chosen process configuration.

In [25]:
def solution_to_dataframe(x, tol=1e-9):
    names = (
        list(barley["name"])
        + list(malts["name"])
        + list(maltextracts["name"])
        + list(hops["name"])
        + list(yeasts["name"])
        + ["z_malt", "z_mash"]
    )
    df = pd.DataFrame({"variable": names, "value": x})
    return df[df["value"].abs() > tol].reset_index(drop=True)

def ebc_from_solution(model, res):
    x = res.x
    b = x[slice_B()]
    m = x[slice_M()]
    e = x[slice_E()]
    m_tot = model["m_tot"]
    gamma = (SG - 1.0) / 0.0344
    return gamma * (
        BARLEY_TO_MALTEXTRACT_EQ * np.sum(barley_ebc * b)
        + MALT_TO_MALTEXTRACT_EQ * np.sum(malt_ebc * m)
        + np.sum(extract_ebc * e)
    ) / m_tot

def print_solution(title, model, res):
    print(f"\n=== {title} ===")
    print("Success:", res.success)
    print("Message:", res.message)
    if not res.success:
        return
    print("Cost:", res.fun)
    print("EBC:", ebc_from_solution(model, res))
    print("z_malt:", res.x[iZm])
    print("z_mash:", res.x[iZs])
    print("\nNon-zero variables:")
    print(solution_to_dataframe(res.x).to_string(index=False))

## Run Task 3.3

This final section solves the concrete problem from the assignment: 5000 litres of amber beer. It computes the relaxation, inspects the fractional process variables, rounds them to a feasible process choice, and then solves the original MILP by brute force.

In [26]:
V_beer = 5000
EBC_min = 20
EBC_max = 29

In [27]:
relax_model, relax_res = solve_relaxation(V_beer, EBC_min, EBC_max)
print_solution("3.3.2 Relaxation", relax_model, relax_res)

rounded_z = rounded_process_choice(relax_res)
print("\nRounded process choice:", rounded_z)

round_model, round_res = solve_with_fixed_processes(V_beer, EBC_min, EBC_max, *rounded_z)
print_solution("3.3.4 Rounded feasible solution", round_model, round_res)

best_solution, all_results = solve_bruteforce(V_beer, EBC_min, EBC_max)
print("\nBrute-force cases:")
print(pd.DataFrame([
    {"z_fix": r["z_fix"], "success": r["success"], "cost": r["cost"]}
    for r in all_results
]).to_string(index=False))

if best_solution is not None:
    print_solution("3.3.5 Original MILP", best_solution["model"], best_solution["result"])


=== 3.3.2 Relaxation ===
Success: True
Message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Cost: 4497.686956521741
EBC: 28.999999999999996
z_malt: 0.0
z_mash: 1.0

Non-zero variables:
variable      value
  malt04 685.348970
  malt06 268.598398
   hop07   6.842105
 yeast05   3.750000
  z_mash   1.000000

Rounded process choice: (0, 1)

=== 3.3.4 Rounded feasible solution ===
Success: True
Message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Cost: 4497.686956521741
EBC: 28.999999999999996
z_malt: -0.0
z_mash: 1.0

Non-zero variables:
variable      value
  malt04 685.348970
  malt06 268.598398
   hop07   6.842105
 yeast05   3.750000
  z_mash   1.000000

Brute-force cases:
 z_fix  success        cost
(0, 0)     True 6776.715526
(0, 1)     True 4497.686957
(1, 1)     True 5469.642544

=== 3.3.5 Original MILP ===
Success: True
Message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Cost: 4497.686956521741
EBC: 28.999999999999996
z_ma

<!-- REPORT_TABLE_OUTPUTS_SYNC -->
## Report table outputs

The following tables correspond to the tables used in the LaTeX report for Task 3.3.

In [28]:
# REPORT_TABLE_OUTPUTS_SYNC
print("Table 6.1: Task 3 Problem Setup")
task33_setup_table = pd.DataFrame([
    {"Parameter": "Beer volume", "Value": "5000 L"},
    {"Parameter": "Colour", "Value": "amber"},
    {"Parameter": "EBC bounds", "Value": "20 to 29"},
    {"Parameter": "Quality constraints", "Value": "not active"},
])
display(task33_setup_table)

print("Table 6.2: Task 3 LP Relaxation Result")
task33_relaxation_result_table = pd.DataFrame([
    {"Item": "Cost", "Result": f"{relax_res.fun:.3f}"},
    {"Item": "$z_{malt}$", "Result": f"{relax_res.x[iZm]:.0f}"},
    {"Item": "$z_{mash}$", "Result": f"{relax_res.x[iZs]:.0f}"},
    {"Item": "Achieved EBC", "Result": f"{ebc_from_solution(relax_model, relax_res):.3f}"},
])
display(task33_relaxation_result_table)

def report_nonzero_ingredients(x, tol=1e-7):
    groups = [
        ("Barley", list(barley["name"]), x[slice_B()]),
        ("Malt", list(malts["name"]), x[slice_M()]),
        ("Malt extract", list(maltextracts["name"]), x[slice_E()]),
        ("Hops", list(hops["name"]), x[slice_H()]),
        ("Yeast", list(yeasts["name"]), x[slice_Y()]),
    ]
    rows = []
    for group, names, values in groups:
        for name, value in zip(names, values):
            if abs(value) > tol:
                rows.append({"Group": group, "Variable": name, "Amount (kg)": value})
    return pd.DataFrame(rows).round({"Amount (kg)": 3})

print("Table 6.3: Task 3 Non-Zero Ingredients in the Relaxed Solution")
task33_relaxation_ingredients_table = report_nonzero_ingredients(relax_res.x)
display(task33_relaxation_ingredients_table)

print("Table 6.4: Task 3 Rounded-Process Re-Solve")
task33_rounded_table = pd.DataFrame([
    {"Item": "Rounded process choice", "Result": str(rounded_z)},
    {"Item": "Re-solved cost", "Result": f"{round_res.fun:.3f}"},
    {"Item": "Achieved EBC", "Result": f"{ebc_from_solution(round_model, round_res):.3f}"},
    {"Item": "Ingredient mix", "Result": "same as Table 6.3"},
])
display(task33_rounded_table)

print("Table 6.5: Task 3 Original MILP Result")
task33_milp_table = pd.DataFrame([
    {"Item": "Selected process choice", "Result": str(best_solution["z_fix"])},
    {"Item": "Total cost", "Result": f"{best_solution['cost']:.3f}"},
    {"Item": "Achieved EBC", "Result": f"{ebc_from_solution(best_solution['model'], best_solution['result']):.3f}"},
    {"Item": "Ingredient mix", "Result": "same as Table 6.3"},
])
display(task33_milp_table)


Table 6.1: Task 3 Problem Setup


,Parameter,Value
0,Beer volume,5000 L
1,Colour,amber
2,EBC bounds,20 to 29
3,Quality constraints,not active


Table 6.2: Task 3 LP Relaxation Result


,Item,Result
0,Cost,4497.687
1,$z_{malt}$,0
2,$z_{mash}$,1
3,Achieved EBC,29.000


Table 6.3: Task 3 Non-Zero Ingredients in the Relaxed Solution


,Group,Variable,Amount (kg)
0,Malt,malt04,685.349
1,Malt,malt06,268.598
2,Hops,hop07,6.842
3,Yeast,yeast05,3.750


Table 6.4: Task 3 Rounded-Process Re-Solve


,Item,Result
0,Rounded process choice,"(0, 1)"
1,Re-solved cost,4497.687
2,Achieved EBC,29.000
3,Ingredient mix,same as Table 6.3


Table 6.5: Task 3 Original MILP Result


,Item,Result
0,Selected process choice,"(0, 1)"
1,Total cost,4497.687
2,Achieved EBC,29.000
3,Ingredient mix,same as Table 6.3
